### BigQuery에서 분석한 SQL 코드

#### 1. 장르별 게임 시장 규모 (어떤 장르의 게임이 가장 많이 분포하고 있는가?)
SELECT TRIM(single_genre) AS genre,
       COUNT(DISTINCT `AppID`) AS game_count,
       ROUND(COUNT(DISTINCT `AppID`) * 100.0 / (SELECT COUNT(*) FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`), 2) AS game_coverage_pct
FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
CROSS JOIN UNNEST(SPLIT(`Genres`, ',')) AS single_genre
WHERE `Genres` IS NOT NULL AND TRIM(single_genre) != ''
GROUP BY genre
ORDER BY game_count DESC
;

#### 2. 장르별 이용자 반응 (장르별 이용자 반응에는 어떤 차이가 있는가?)
SELECT TRIM(single_genre) AS genre,
       COUNT(DISTINCT `AppID`) AS reliable_game_count,
       ROUND(AVG(SAFE_DIVIDE(`Positive`, `Positive` + `Negative`)) * 100, 2) AS avg_positive_rate_pct,
       ROUND(AVG(`Peak CCU`), 1) AS avg_peak_ccu,
       MAX(`Peak CCU`) AS max_peak_ccu
FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
CROSS JOIN UNNEST(SPLIT(`Genres`, ',')) AS single_genre
WHERE `Genres` IS NOT NULL AND TRIM(single_genre) != '' AND (`Positive` + `Negative`) >= 30
GROUP BY genre
HAVING reliable_game_count >= 10
ORDER BY avg_positive_rate_pct DESC
;

#### 3. 가격대별 이용자 반응 (게임 가격대에 따라 이용자 반응이 달라지는가?)
SELECT CASE WHEN `Price` = 0 THEN '1. Free'
            WHEN `Price` > 0 AND `Price` <= 10 THEN '2. $1~$10'
            WHEN `Price` > 10 AND `Price` <= 20 THEN '3. $10~$20'
            WHEN `Price` > 20 AND `Price` <= 40 THEN '4. $20~$40'
            ELSE '5. $40+' END AS price_group,
       COUNT(*) AS total_games,
       ROUND(AVG(SAFE_DIVIDE(`Positive`, `Positive` + `Negative`)) * 100, 2) AS avg_positive_rate_pct,
       ROUND(AVG(`Peak CCU`), 1) AS avg_peak_ccu,
       ROUND(AVG(`Average playtime forever` / 60.0), 2) AS avg_playtime_hours
FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
WHERE (`Positive` + `Negative`) >= 30
GROUP BY price_group
ORDER BY price_group
;

#### 4. 가격과 이용자 규모 간의 관계 (게임 가격과 이용자 규모 사이에는 어떤 관계가 있는가?)
SELECT `AppID`, `Name`, `Price`, `Peak CCU`, (`Positive` + `Negative`) AS total_reviews,
       ROUND(SAFE_DIVIDE(`Positive`, `Positive` + `Negative`) * 100, 2) AS positive_rate_pct,
      `Estimated owners`
FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
WHERE `Peak CCU` > 0
ORDER BY `Peak CCU` DESC
LIMIT 1000
;

#### 5. 게임 연령과 최근 이용자 활동 (출시 후 시간이 오래 지난 게임도 최근 이용자 활동을 유지하고 있는가?)
WITH parsed_age_data AS (SELECT `AppID`, `Name`,
                         2026 - EXTRACT(YEAR FROM COALESCE(SAFE.PARSE_DATE('%b %d, %Y', `Release date`),
                                                           SAFE.PARSE_DATE('%Y-%m-%d', `Release date`),
                                                           SAFE.PARSE_DATE('%b %Y', `Release date`))) AS game_age,
                        `Average playtime two weeks` / 60.0 AS recent_playtime_hours, `Peak CCU`
                        FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
                        WHERE `Release date` IS NOT NULL AND `Average playtime forever` > 0)
SELECT CASE WHEN game_age < 1 THEN '1. 0~1년 (최신작)'
            WHEN game_age BETWEEN 1 AND 3 THEN '2. 1~3년'
            WHEN game_age BETWEEN 3 AND 5 THEN '3. 3~5년'
            WHEN game_age BETWEEN 5 AND 10 THEN '4. 5~10년'
            ELSE '5. 10년 이상 (클래식)' END AS game_age_group,
       COUNT(*) AS active_game_count,
       ROUND(AVG(recent_playtime_hours), 2) AS avg_recent_playtime_hours,
       ROUND(AVG(`Peak CCU`), 1) AS avg_peak_ccu
FROM parsed_age_data
WHERE game_age >= 0
GROUP BY game_age_group
ORDER BY game_age_group
;

#### 6. DLC 규모와 이용자 참여도 (추가 콘텐츠(DLC) 규모는 이용자 참여와 관련이 있는가?)
SELECT CASE WHEN `DLC count` = 0 THEN '1. DLC 없음 (0개)'
            WHEN `DLC count` BETWEEN 1 AND 2 THEN '2. 소규모 (1~2개)'
            WHEN `DLC count` BETWEEN 3 AND 5 THEN '3. 중규모 (3~5개)'
            ELSE '4. 대규모 (6개 이상)' END AS dlc_scale_group,
       COUNT(*) AS total_games,
       ROUND(AVG(`Average playtime forever` / 60.0), 2) AS avg_total_playtime_hours,
       ROUND(AVG(`Average playtime two weeks` / 60.0), 2) AS avg_recent_playtime_hours,
       ROUND(AVG(`Peak CCU`), 1) AS avg_peak_ccu
FROM `steam-analysis-project-2026.steam_analysis.steam_analysis_2026`
WHERE `Average playtime forever` > 0
GROUP BY dlc_scale_group
ORDER BY dlc_scale_group
;

In [4]:
!pip install google-cloud-bigquery google-cloud-storage pandas pyarrow db-dtypes

Defaulting to user installation because normal site-packages is not writeable
     -------------------------------------- 262.3/262.3 kB 5.4 MB/s eta 0:00:00
     -------------------------------------- 321.3/321.3 kB 6.6 MB/s eta 0:00:00
     --------------------------------------- 26.2/26.2 MB 10.9 MB/s eta 0:00:00
  Using cached packaging-26.3-py3-none-any.whl (129 kB)
     ---------------------------------------- 81.5/81.5 kB ? eta 0:00:00
     ------------------------------------- 173.3/173.3 kB 10.2 MB/s eta 0:00:00
     ------------------------------------- 246.5/246.5 kB 14.8 MB/s eta 0:00:00
     ---------------------------------------- 11.4/11.4 MB 9.6 MB/s eta 0:00:00
     --------------------------------------- 15.9/15.9 MB 12.8 MB/s eta 0:00:00
  Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)
     -------------------------------------- 300.6/300.6 kB 4.7 MB/s eta 0:00:00
     ---------------------------------------- 50.5/50.5 kB ? eta 0:00:00
     ----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
anaconda-project 0.11.1 requires ruamel-yaml, which is not installed.
selenium 4.36.0 requires urllib3[socks]<3.0,>=2.5.0, but you have urllib3 1.26.20 which is incompatible.
scipy 1.9.1 requires numpy<1.25.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 2.0.2 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.


In [1]:
from google.cloud import bigquery
import pandas as pd

# gcloud 로그인 토큰을 자동 감지하므로 키 파일 경로 설정이 필요 없습니다.
project_id = "steam-analysis-project-2026"
client = bigquery.Client(project=project_id)

query = f"""
SELECT *
FROM `{project_id}.steam_analysis.steam_analysis_2026`
"""

print("BigQuery 데이터 로드 중...")
df = client.query(query).to_dataframe()
print(f"로드 완료! 데이터 크기: {df.shape[0]:,}행, {df.shape[1]}열")
df.head(3)

C:\Users\wjdtm\AppData\Roaming\Python\Python39\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\wjdtm\AppData\Roaming\Python\Python39\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\wjdtm\AppData\Roaming\Python\Python39\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\ProgramData\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\ProgramData\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\ProgramData\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

BigQuery 데이터 로드 중...


C:\Users\wjdtm\AppData\Roaming\Python\Python39\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


로드 완료! 데이터 크기: 114,172행, 38열


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots
0,2025300,I’m going to die if I don’t eat sushi!,"Jun 3, 2022",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,https://shared.akamai.steamstatic.com/store_it...
1,3044050,DOA - Private Testing,"May 15, 2025",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,None
2,1750090,Knight&Princess,"Oct 24, 2021",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,None


In [2]:
# 라이브러리 호출
import pandas as pd
import numpy as np

In [3]:
# 데이터 확인하기
df.shape

(114172, 38)

In [4]:
df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots
0,2025300,I’m going to die if I don’t eat sushi!,"Jun 3, 2022",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,https://shared.akamai.steamstatic.com/store_it...
1,3044050,DOA - Private Testing,"May 15, 2025",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,None
2,1750090,Knight&Princess,"Oct 24, 2021",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,None
3,3112490,Death Report,"Nov 1, 2024",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,https://shared.akamai.steamstatic.com/store_it...
4,451330,IS Defense Editor,"Apr 29, 2016",0 - 0,0,0,0.0,0,0,None,...,0,0,0,0,None,None,None,None,None,None


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114172 entries, 0 to 114171
Data columns (total 38 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       114172 non-null  Int64  
 1   Name                        114172 non-null  object 
 2   Release date                114172 non-null  object 
 3   Estimated owners            114172 non-null  object 
 4   Peak CCU                    114172 non-null  Int64  
 5   Required age                114172 non-null  Int64  
 6   Price                       114172 non-null  float64
 7   Discount                    114172 non-null  Int64  
 8   DLC count                   114172 non-null  Int64  
 9   About the game              113902 non-null  object 
 10  Supported languages         114172 non-null  object 
 11  Full audio languages        114172 non-null  object 
 12  Reviews                     12029 non-null   object 
 13  Header image  

In [6]:
# 날짜 변환 및 게임 연령 파생
df['Release date'] = pd.to_datetime(df['Release date'], errors='coerce')
df['release_year'] = df['Release date'].dt.year
df['game_age'] = 2026 - df['release_year']

In [7]:
# 게임 연령 그룹 생성
age_bins = [-np.inf, 1, 3, 5, 10, np.inf]
age_labels = ['0~1년', '1~3년', '3~5년', '5~10년', '10년+']
df['game_age_group'] = pd.cut(df['game_age'], bins=age_bins, labels=age_labels, right=False)

In [8]:
# 긍정 리뷰율 계산 (리뷰 수 0건 예외 처리)
total_reviews = df['Positive'] + df['Negative']
df['positive_rate'] = np.where(total_reviews > 0, df['Positive'] / total_reviews, np.nan)

In [9]:
# 가격 구간 생성
price_bins = [-np.inf, 0, 10, 20, 40, np.inf]
price_labels = ['Free', '$1~$10', '$10~$20', '$20~$40', '$40+']
df['price_group'] = pd.cut(df['Price'], bins=price_bins, labels=price_labels, right=True)

In [10]:
# 지원 플랫폼 개수
df['platform_count'] = (df['Windows'].astype(int) + df['Mac'].astype(int) + df['Linux'].astype(int))

In [11]:
# 장르 결측치 처리
df['Genres'] = df['Genres'].fillna('Unknown')

In [12]:
# 컬럼명 Snake Case 및 소문자 변환
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

In [13]:
# 플레이타임 분 -> 시간 변환
df["playtime_forever_hours"] = (df["average_playtime_forever"] / 60).round(1)
df["playtime_2weeks_hours"] = (
    df["average_playtime_two_weeks"] / 60
).round(1)

In [14]:
# 리뷰 지표 및 신뢰성 플래그 (True / False)
df["total_reviews"] = df["positive"] + df["negative"]
df["positive_rate"] = np.where(df["total_reviews"] > 0, df["positive"] / df["total_reviews"], np.nan)
df["is_reliable_review"] = df["total_reviews"] >= 30

In [15]:
# 플랫폼 개수 및 장르 결측치 처리
df["platform_count"] = (df["windows"].astype(int) + df["mac"].astype(int) + df["linux"].astype(int))
df["genres"] = df["genres"].fillna("Unknown")

In [16]:
df.head()

,appid,name,release_date,estimated_owners,peak_ccu,required_age,price,discount,dlc_count,about_the_game,...,release_year,game_age,game_age_group,positive_rate,price_group,platform_count,playtime_forever_hours,playtime_2weeks_hours,total_reviews,is_reliable_review
0,2025300,I’m going to die if I don’t eat sushi!,2022-06-03,0 - 0,0,0,0.0,0,0,None,...,2022,4,3~5년,NaN,Free,1,0.0,0.0,0,False
1,3044050,DOA - Private Testing,2025-05-15,0 - 0,0,0,0.0,0,0,None,...,2025,1,1~3년,NaN,Free,1,0.0,0.0,0,False
2,1750090,Knight&Princess,2021-10-24,0 - 0,0,0,0.0,0,0,None,...,2021,5,5~10년,NaN,Free,1,0.0,0.0,0,False
3,3112490,Death Report,2024-11-01,0 - 0,0,0,0.0,0,0,None,...,2024,2,1~3년,NaN,Free,1,0.0,0.0,0,False
4,451330,IS Defense Editor,2016-04-29,0 - 0,0,0,0.0,0,0,None,...,2016,10,10년+,NaN,Free,1,0.0,0.0,0,False


In [18]:
# 원본(분 단위)과 변환된 컬럼(시간 단위) 비교 조회
playtime_cols = [
    'name',
    'average_playtime_forever',
    'playtime_forever_hours',
    'average_playtime_two_weeks',
    'playtime_2weeks_hours',
]
df[playtime_cols]

,name,average_playtime_forever,playtime_forever_hours,average_playtime_two_weeks,playtime_2weeks_hours
0,I’m going to die if I don’t eat sushi!,0,0.0,0,0.0
1,DOA - Private Testing,0,0.0,0,0.0
2,Knight&Princess,0,0.0,0,0.0
3,Death Report,0,0.0,0,0.0
4,IS Defense Editor,0,0.0,0,0.0
...,...,...,...,...,...
114167,Marine Radar Simulator - VR,0,0.0,0,0.0
114168,Cyberlink PowerDVD 17 Ultra,1,0.0,0,0.0
114169,Museum of Science Fiction: Experimental VR Gal...,0,0.0,0,0.0
114170,EasyMMD,0,0.0,0,0.0


In [19]:
# 1시간 미만(playtime_forever_hours < 1.0) 게임 필터링
under_1h = df['playtime_forever_hours'] < 1.0
under_1h_count = under_1h.sum()
under_1h_ratio = (under_1h_count / len(df)) * 100

print(f'전체 게임 수: {len(df):,}개')
print(f'누적 플레이타임 1시간 미만 게임 수: {under_1h_count:,}개')
print(f'1시간 미만 게임 비중: {under_1h_ratio:.2f}%')

전체 게임 수: 114,172개
누적 플레이타임 1시간 미만 게임 수: 93,010개
1시간 미만 게임 비중: 81.46%


In [21]:
# 세부 구간 조건
zero_hours = (df['playtime_forever_hours'] == 0).sum()
under_1h_active = (
    (df['playtime_forever_hours'] > 0) & (df['playtime_forever_hours'] < 1.0)
).sum()
over_1h = (df['playtime_forever_hours'] >= 1.0).sum()
total_len = len(df)

# 요약 데이터프레임 생성 (문자열 포맷팅 적용)
playtime_summary = pd.DataFrame(
    {
        '구간': ['기록 없음 (0시간)', '0시간 초과 ~ 1시간 미만', '1시간 이상'],
        '게임 수': [
            f'{zero_hours:,}개',
            f'{under_1h_active:,}개',
            f'{over_1h:,}개',
        ],
        '비율 (%)': [
            f'{(zero_hours / total_len) * 100:.2f}%',
            f'{(under_1h_active / total_len) * 100:.2f}%',
            f'{(over_1h / total_len) * 100:.2f}%',
        ],
    }
)

playtime_summary

,구간,게임 수,비율 (%)
0,기록 없음 (0시간),"88,726개",77.71%
1,0시간 초과 ~ 1시간 미만,"4,284개",3.75%
2,1시간 이상,"21,162개",18.54%


In [22]:
# 플레이 기록 유무 플래그 생성
df['has_playtime'] = df['playtime_forever_hours'] > 0

In [23]:
df.head()

,appid,name,release_date,estimated_owners,peak_ccu,required_age,price,discount,dlc_count,about_the_game,...,game_age,game_age_group,positive_rate,price_group,platform_count,playtime_forever_hours,playtime_2weeks_hours,total_reviews,is_reliable_review,has_playtime
0,2025300,I’m going to die if I don’t eat sushi!,2022-06-03,0 - 0,0,0,0.0,0,0,None,...,4,3~5년,NaN,Free,1,0.0,0.0,0,False,False
1,3044050,DOA - Private Testing,2025-05-15,0 - 0,0,0,0.0,0,0,None,...,1,1~3년,NaN,Free,1,0.0,0.0,0,False,False
2,1750090,Knight&Princess,2021-10-24,0 - 0,0,0,0.0,0,0,None,...,5,5~10년,NaN,Free,1,0.0,0.0,0,False,False
3,3112490,Death Report,2024-11-01,0 - 0,0,0,0.0,0,0,None,...,2,1~3년,NaN,Free,1,0.0,0.0,0,False,False
4,451330,IS Defense Editor,2016-04-29,0 - 0,0,0,0.0,0,0,None,...,10,10년+,NaN,Free,1,0.0,0.0,0,False,False


#### 전처리 내용 정리
- 컬럼명 표준화
    - 모든 컬럼명을 소문자 및 Snake Case(_) 형식으로 변환하여 SQL 작성 및 데이터 접근 편의성 확보

- 날짜 변환 및 게임 연령 파생변수 생성
    - 문자열 형태의 release_date를 날짜형(datetime)으로 변환 후 release_year 추출
    - 기준연도(2026년) 기반 게임 연령(game_age) 및 연령대 구간(game_age_group: 0~1년, 1~3년, 3~5년, 5~10년, 10년+) 생성

- 리뷰 지표 정제 및 신뢰성 플래그 추가
    - 총 리뷰 수(total_reviews = positive + negative) 및 긍정 리뷰율(positive_rate) 계산 (리뷰 0건 예외 처리)
    -소수 리뷰로 인한 통계 왜곡 방지를 위해 유효 리뷰 여부 플래그(is_reliable_review: 총 리뷰 30개 이상) 생성

- 가격 구간 범주화
    - price 컬럼을 분석용 5개 구간(price_group: Free, $1~$10, $10~$20, $20~$40, $40+)으로 범주화

- 플레이타임 단위 변환 및 활성 유저 플래그 생성
    - 분(Minute) 단위의 플레이타임 컬럼을 시간(Hours) 단위(playtime_forever_hours, playtime_2weeks_hours)로 변환
    - 누적 플레이타임 기록 유무에 따른 분석 분리를 위해 활성 플레이 플래그(has_playtime: playtime > 0) 생성

- 플랫폼 및 결측치 처리
    - OS 지원 여부(windows, mac, linux)를 합산하여 지원 플랫폼 수(platform_count) 생성
    - genres 컬럼의 결측치를 'Unknown'으로 대체하여 필터링 및 집계 오류 방지